In [1]:
# Load preprocessed and augmented data from 01_preprocessing.ipynb
%run 01_preprocessing.ipynb

Since the GPL-licensed package `unidecode` is not installed, using Python's `unicodedata` package which yields worse results.



Data augmentation complete!
Original rows (body only): 2029
Augmented rows (body + examples): 8310
New rows added: 6281

Label distribution:
rule_violation
1    4229
0    4081
Name: count, dtype: int64

Columns in new df (examples removed): ['body', 'rule', 'subreddit', 'rule_violation', 'source']

Applied combine_comment_rule to augmented dataset
Final columns after augmentation and unique comment-rule pair combination: ['body', 'rule', 'subreddit', 'rule_violation', 'source', 'combined_text']
Total rows after cleaning: 8310


In [2]:
from preprocessing import write_positive_negative_label, split_data, map_fasttext_labels
from models.fasttext import write_fasttext_file, fasttext_predict
from eval import evaluate_model
import fasttext

In [3]:
train_df_clean = write_positive_negative_label(train_df_clean)

In [ ]:
train_df_clean.head()

In [4]:
# Train and validation split (on fasttext labelled data)
train_df, val_df = split_data(train_df_clean, label_column='rule_violation')

In [5]:
write_fasttext_file(
    df=train_df,
    text_col="combined_text",
    label_col="rule_violation",
    out_path="../data/fasttext/fasttext_train.txt"
)
write_fasttext_file(
    df=val_df,
    text_col="combined_text",
    label_col="rule_violation",
    out_path="../data/fasttext/fasttext_val.txt"
)

In [ ]:
for i in range(5):
    print(repr(open("../data/fasttext/fasttext_train.txt", "r").readline()))

In [6]:
model = fasttext.train_supervised(
    input="../data/fasttext/fasttext_train.txt",
    autotuneMetric='f1:__label__1'
)

Read 0M words
Number of words:  11030
Number of labels: 2
Progress: 100.0% words/sec/thread: 1454353 lr:  0.000000 avg.loss:  0.445144 ETA:   0h 0m 0s


In [7]:
import random

with open("../data/fasttext/fasttext_train.txt", "r") as f:
    lines = f.readlines()

random.sample(lines, 20)

['__label__0 [WTS] keys Windows 7/8/8.1/10 | Windows Server | Office | Visio | Project <URL> [SEP] No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.\n',
 '__label__1 FindSexToday . com - free girls for sex worldwide 25Rn9YwSYYyw18! [SEP] No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.\n',
 '__label__1 Ray-Ban Sunglasses Special Sale at $24.99 Shop Now >>> <URL> [SEP] No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.\n',
 '__label__0 **HD** [ German Stream ](<URL>) | Mobile : Yes [SEP] No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.\n',
 "__label__0 Banks don't want you to know this! Click here to know more! [SEP] No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.\n",
 '__label__1 Ray-Ban Sunglasses Special Sale at $24

In [8]:
model.save_model("fasttext_baseline.bin")

In [10]:
# Load model
model = fasttext.load_model("fasttext_baseline.bin")

# Extract validation text
X_val_texts = val_df["combined_text"].tolist()

# Predictx
y_pred_labels, y_pred_probs = fasttext_predict(model, X_val_texts)

ValueError: Unable to avoid copy while creating an array as requested.
If using `np.array(obj, copy=False)` replace it with `np.asarray(obj)` to allow a copy when needed (no behavior change in NumPy 1.x).
For more details, see https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword.

In [ ]:
val_df[val_df["combined_text"].str.strip().str.len() == 0]

In [ ]:
val_df["combined_text"].isna().sum()

In [ ]:
# Evaluate
evaluate_model(y_val, y_pred_labels, y_pred_probs, title="FastText Baseline")